# Model-family walkthrough

This notebook presents the 14 pre-registered model trials used by the benchmark. Each model has one executable cell, and each cell is preceded by a short explanation.

The families are grouped in a fixed order. Within each family, variants run from smaller to larger capacity. Every model uses the shared fixed raw split, selects its best epoch on validation macro-F1, saves a resumable checkpoint, and defers the held-out test.

Set COMPCLASS_PROFILE=smoke for a fast pipeline check. Set COMPCLASS_WALKTHROUGH_ARTIFACT_PREFIX to choose a new checkpoint prefix; the default family_walkthrough does not overwrite earlier benchmark artifacts.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_DIR = Path(os.getenv("COMPCLASS_WORK_DIR", Path.cwd())).expanduser().resolve()
NOTEBOOK_DIR = WORK_DIR if (WORK_DIR / "12_model_family_benchmark.py").exists() else WORK_DIR / "notebooks"
RUNNER_PATH = NOTEBOOK_DIR / "12_model_family_benchmark.py"
if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Could not find {RUNNER_PATH}")

PROFILE_DEFAULTS = {"smoke": (1, 1), "standard": (10, 3), "paper": (25, 5)}
PROFILE = os.getenv("COMPCLASS_PROFILE", "standard")
if PROFILE not in PROFILE_DEFAULTS:
    raise ValueError(f"Unknown COMPCLASS_PROFILE={PROFILE!r}")
EPOCHS = int(os.getenv("COMPCLASS_WALKTHROUGH_EPOCHS", PROFILE_DEFAULTS[PROFILE][0]))
PATIENCE = int(os.getenv("COMPCLASS_WALKTHROUGH_PATIENCE", PROFILE_DEFAULTS[PROFILE][1]))
RECIPE = os.getenv("COMPCLASS_WALKTHROUGH_RECIPE", "robust-v2-tta")
ARTIFACT_PREFIX = os.getenv("COMPCLASS_WALKTHROUGH_ARTIFACT_PREFIX", "family_walkthrough")
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"

MODEL_GROUPS = {
    "ResNet": ["resnet34", "resnet101", "resnet152"],
    "DenseNet": ["densenet121"],
    "EfficientNet": ["efficientnet_b2", "efficientnet_v2_s", "efficientnet_v2_m"],
    "MobileNet": ["mobilenet_v3_large"],
    "ConvNeXt": ["convnext_tiny", "convnext_small", "convnext_base"],
    "Swin": ["swin_t", "swin_v2_t", "swin_b"],
}
MODEL_ORDER = [name for names in MODEL_GROUPS.values() for name in names]
MODEL_FAMILY = {name: family for family, names in MODEL_GROUPS.items() for name in names}
MODEL_RECORDS = {}
REPORT_PREFIX = f"model_walkthrough_{RECIPE.replace('-', '_')}"

def _report_path(model):
    return METRICS_DIR / f"{REPORT_PREFIX}_{model}.json"

def _row(record):
    training = record.get("training", {})
    validation = record.get("val", {})
    return {
        "family": MODEL_FAMILY[record["name"]],
        "model": record["name"],
        "status": "complete",
        "parameters": record.get("params", training.get("params", float("nan"))),
        "best_epoch": training.get("best_epoch", float("nan")),
        "val_accuracy": validation.get("accuracy", float("nan")),
        "val_macro_f1": validation.get("macro_f1", float("nan")),
        "val_weighted_f1": validation.get("weighted_f1", float("nan")),
        "train_accuracy": training.get("best_train_accuracy", float("nan")),
        "train_val_gap": training.get("train_val_accuracy_gap", float("nan")),
        "overfit_flag": record.get("overfit_flag", False),
        "wall_time_seconds": training.get("wall_time_seconds", float("nan")),
        "checkpoint": Path(record.get("checkpoint", "")).name,
    }

def _format_number(value, digits=4):
    return "—" if pd.isna(value) else f"{value:.{digits}f}"

def _display_record(record):
    row = pd.DataFrame([_row(record)])
    display(row.style.format({
        "parameters": lambda value: "—" if pd.isna(value) else f"{int(value):,}",
        "best_epoch": lambda value: "—" if pd.isna(value) else f"{int(value)}",
        "val_accuracy": _format_number,
        "val_macro_f1": _format_number,
        "val_weighted_f1": _format_number,
        "train_accuracy": _format_number,
        "train_val_gap": _format_number,
        "wall_time_seconds": lambda value: "—" if pd.isna(value) else f"{value:.1f}",
    }).background_gradient(subset=["val_accuracy", "val_macro_f1"], cmap="YlGn"))

def run_model(model):
    if model not in MODEL_ORDER:
        raise ValueError(f"Unknown walkthrough model: {model}")
    report_name = _report_path(model).name
    command = [
        sys.executable, str(RUNNER_PATH),
        "--models", model,
        "--profile", PROFILE,
        "--epochs", str(EPOCHS),
        "--patience", str(PATIENCE),
        "--recipe", RECIPE,
        "--artifact-prefix", ARTIFACT_PREFIX,
        "--report", report_name,
        "--defer-test",
    ]
    print(f"Running {model} | recipe={RECIPE} | profile={PROFILE} | report={report_name}")
    subprocess.run(
        command,
        cwd=NOTEBOOK_DIR,
        check=True,
        env={**os.environ, "COMPCLASS_WORK_DIR": str(NOTEBOOK_DIR)},
    )
    payload = json.loads(_report_path(model).read_text())
    trials = [trial for trial in payload.get("trials", []) if trial.get("name") == model]
    if len(trials) != 1:
        raise RuntimeError(f"Expected one completed trial for {model} in {_report_path(model)}")
    record = trials[0]
    MODEL_RECORDS[model] = record
    _display_record(record)
    return record

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Recipe: {RECIPE} | profile: {PROFILE} | epochs: {EPOCHS} | patience: {PATIENCE}")
print(f"Models: {len(MODEL_ORDER)} | held-out test: deferred")

Notebook directory: /home/vaper/Work/Prototypes/626-Project/notebooks
Recipe: robust-v2-tta | profile: standard | epochs: 10 | patience: 3
Models: 14 | held-out test: deferred


## ResNet family

ResNet is a convolutional family built around residual skip connections, which let deep networks learn stable refinements.

**Pros:** mature transfer-learning behavior, stable optimization, and clear depth scaling.

**Cons:** deeper variants are expensive, can memorize noisy images, and mainly model local visual patterns.

### ResNet-34 — compact residual CNN

This smaller residual candidate establishes a lower-capacity baseline and tests the gain from moving beyond ResNet-18.

In [2]:
run_model("resnet34")

Running resnet34 | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_resnet34.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/ou

100%|██████████| 83.3M/83.3M [00:07<00:00, 11.7MB/s]
                                                                      

resnet34 epoch=01 loss=2.8660 train_acc=0.1731 val_acc=0.2471 val_macro_f1=0.2272


resnet34 epoch=02 loss=2.1680 train_acc=0.4360 val_acc=0.5038 val_macro_f1=0.4898


resnet34 epoch=03 loss=1.5475 train_acc=0.6489 val_acc=0.5785 val_macro_f1=0.5671


resnet34 epoch=04 loss=1.1763 train_acc=0.7892 val_acc=0.6284 val_macro_f1=0.6093


... (2 outputs left out)

resnet34 epoch=07 loss=0.7406 train_acc=0.9495 val_acc=0.6513 val_macro_f1=0.6300


resnet34 epoch=08 loss=0.7001 train_acc=0.9692 val_acc=0.6533 val_macro_f1=0.6306
{
  "model": "resnet34",
  "val": {
    "accuracy": 0.6590038314176245,
    "macro_f1": 0.6316920597586609,
    "weighted_f1": 0.6547019057289211
  },
  "gap": 0.22532758777843787,
  "overfit_flag": false
}
{
  "selected_model": "resnet34",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ResNet,resnet34,complete,"21,294,932",5,0.6590,0.6317,0.6547,0.8843,0.2253,False,381.2,family_walkthrough_resnet34.pt


{'name': 'resnet34',
 'settings': {'family': 'resnet',
  'backbone': 'resnet34',
  'lr': 0.001,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 21294932,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.865990983115302,
    'train_accuracy': 0.17309269893355209,
    'val_accuracy': 0.2471264367816092,
    'val_macro_f1': 0.22723448174220984,
    'val_weighted_f1': 0.23918914465881835},
   {'epoch': 2,
    'train_loss': 2.1679553019455056,
    'train_accuracy': 0.4360131255127153,
    'val_accuracy': 0.5038314176245211,
    'val_macro_f1': 0.48983997990884526,
    'val_weighted_f1': 0.5032206831881899},
   {'epoch': 3,
    'train_loss': 1.547547493884766,
    'train_accuracy': 0.6488925348646432,
    'val_accuracy': 0

### ResNet-101 — deep residual CNN

This deep residual candidate tests whether deeper hierarchical features resolve ambiguous component views better than compact ResNets.

In [3]:
run_model("resnet101")

Running resnet101 | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_resnet101.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/

100%|██████████| 171M/171M [00:17<00:00, 10.1MB/s] 
                                                                       

resnet101 epoch=01 loss=2.6880 train_acc=0.2867 val_acc=0.3985 val_macro_f1=0.3832


resnet101 epoch=02 loss=1.9742 train_acc=0.5148 val_acc=0.5594 val_macro_f1=0.5435


resnet101 epoch=03 loss=1.4241 train_acc=0.7055 val_acc=0.6590 val_macro_f1=0.6440


resnet101 epoch=04 loss=1.0793 train_acc=0.8343 val_acc=0.6992 val_macro_f1=0.6840


... (4 outputs left out)

resnet101 epoch=09 loss=0.5975 train_acc=0.9922 val_acc=0.7165 val_macro_f1=0.6960


resnet101 epoch=10 loss=0.5912 train_acc=0.9918 val_acc=0.7280 val_macro_f1=0.7060
{
  "model": "resnet101",
  "val": {
    "accuracy": 0.7298850574712644,
    "macro_f1": 0.7100391732623846,
    "weighted_f1": 0.7271210499447128
  },
  "gap": 0.2561690852686863,
  "overfit_flag": false
}
{
  "selected_model": "resnet101",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ResNet,resnet101,complete,"42,541,140",8,0.7299,0.7100,0.7271,0.9861,0.2562,False,652.3,family_walkthrough_resnet101.pt


{'name': 'resnet101',
 'settings': {'family': 'resnet',
  'backbone': 'resnet101',
  'lr': 0.0007,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 42541140,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.687976399278329,
    'train_accuracy': 0.2867104183757178,
    'val_accuracy': 0.39846743295019155,
    'val_macro_f1': 0.3832261360296535,
    'val_weighted_f1': 0.3948982656608928},
   {'epoch': 2,
    'train_loss': 1.974195570727579,
    'train_accuracy': 0.514766201804758,
    'val_accuracy': 0.5593869731800766,
    'val_macro_f1': 0.5434718918970718,
    'val_weighted_f1': 0.5559000185227386},
   {'epoch': 3,
    'train_loss': 1.4240587864046783,
    'train_accuracy': 0.7054963084495488,
    'val_accuracy': 0.

### ResNet-152 — largest residual CNN

This largest ResNet probes the upper end of residual capacity while validation and the gap guard protect against memorising noise.

In [4]:
run_model("resnet152")

Running resnet152 | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_resnet152.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/

100%|██████████| 230M/230M [00:22<00:00, 11.0MB/s] 
                                                                       

resnet152 epoch=01 loss=2.7482 train_acc=0.2596 val_acc=0.3812 val_macro_f1=0.3592


resnet152 epoch=02 loss=2.0531 train_acc=0.4906 val_acc=0.5402 val_macro_f1=0.5234


resnet152 epoch=03 loss=1.5029 train_acc=0.6801 val_acc=0.6226 val_macro_f1=0.6046


resnet152 epoch=04 loss=1.1629 train_acc=0.7982 val_acc=0.6667 val_macro_f1=0.6452


... (4 outputs left out)

resnet152 epoch=09 loss=0.6086 train_acc=0.9906 val_acc=0.7241 val_macro_f1=0.7040


resnet152 epoch=10 loss=0.6008 train_acc=0.9885 val_acc=0.7203 val_macro_f1=0.7033
{
  "model": "resnet152",
  "val": {
    "accuracy": 0.7241379310344828,
    "macro_f1": 0.7039986449316586,
    "weighted_f1": 0.7166110955806307
  },
  "gap": 0.2664281067013663,
  "overfit_flag": false
}
{
  "selected_model": "resnet152",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ResNet,resnet152,complete,"58,184,788",9,0.7241,0.7040,0.7166,0.9906,0.2664,False,775.9,family_walkthrough_resnet152.pt


{'name': 'resnet152',
 'settings': {'family': 'resnet',
  'backbone': 'resnet152',
  'lr': 0.0005,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 58184788,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.7481646974102345,
    'train_accuracy': 0.25963904840032814,
    'val_accuracy': 0.38122605363984674,
    'val_macro_f1': 0.3591544174459119,
    'val_weighted_f1': 0.36807636619157325},
   {'epoch': 2,
    'train_loss': 2.05311630364337,
    'train_accuracy': 0.49056603773584906,
    'val_accuracy': 0.5402298850574713,
    'val_macro_f1': 0.523449718158184,
    'val_weighted_f1': 0.5304004162522369},
   {'epoch': 3,
    'train_loss': 1.5029051896014245,
    'train_accuracy': 0.6800656275635767,
    'val_accuracy':

## DenseNet family

DenseNet connects each layer to later layers inside dense blocks, encouraging feature reuse instead of relearning features.

**Pros:** parameter-efficient feature reuse and strong fine-grained feature propagation.

**Cons:** dense concatenation increases memory traffic, training can be slower, and scaling is less straightforward.

### DenseNet-121 — dense feature reuse

This moderate-size candidate tests whether dense feature reuse helps more than residual depth on visually similar components.

In [5]:
run_model("densenet121")

Running densenet121 | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_densenet121.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebo

100%|██████████| 30.8M/30.8M [00:02<00:00, 11.6MB/s]
                                                                         

densenet121 epoch=01 loss=2.7925 train_acc=0.1993 val_acc=0.3027 val_macro_f1=0.2740


densenet121 epoch=02 loss=2.1361 train_acc=0.4631 val_acc=0.5192 val_macro_f1=0.4987


densenet121 epoch=03 loss=1.6367 train_acc=0.6382 val_acc=0.5805 val_macro_f1=0.5543


densenet121 epoch=04 loss=1.3543 train_acc=0.7326 val_acc=0.6418 val_macro_f1=0.6169


... (4 outputs left out)

densenet121 epoch=09 loss=0.6718 train_acc=0.9840 val_acc=0.6762 val_macro_f1=0.6500


densenet121 epoch=10 loss=0.6364 train_acc=0.9869 val_acc=0.6724 val_macro_f1=0.6389
{
  "model": "densenet121",
  "val": {
    "accuracy": 0.6877394636015326,
    "macro_f1": 0.6665090017045968,
    "weighted_f1": 0.6874777409853096
  },
  "gap": 0.27616537643128114,
  "overfit_flag": false
}
{
  "selected_model": "densenet121",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,DenseNet,densenet121,complete,"6,974,356",8,0.6877,0.6665,0.6875,0.9639,0.2762,False,560.2,family_walkthrough_densenet121.pt


{'name': 'densenet121',
 'settings': {'family': 'densenet',
  'backbone': 'densenet121',
  'lr': 0.001,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 6974356,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.792463710884643,
    'train_accuracy': 0.19934372436423298,
    'val_accuracy': 0.30268199233716475,
    'val_macro_f1': 0.2739869164530304,
    'val_weighted_f1': 0.2739341414441038},
   {'epoch': 2,
    'train_loss': 2.1361244418262655,
    'train_accuracy': 0.463084495488105,
    'val_accuracy': 0.5191570881226054,
    'val_macro_f1': 0.49874052741471236,
    'val_weighted_f1': 0.5114876580684103},
   {'epoch': 3,
    'train_loss': 1.6366593923444062,
    'train_accuracy': 0.638228055783429,
    'val_accurac

## EfficientNet family

EfficientNet scales depth, width, and input resolution together; EfficientNet-V2 modernizes that training and scaling recipe.

**Pros:** strong accuracy per parameter and useful capacity steps.

**Cons:** performance can be sensitive to recipe and resolution, while large variants become memory- and time-intensive.

### EfficientNet-B2 — efficient CNN baseline

This smaller EfficientNet tests whether compound scaling gives better accuracy per parameter than the other CNN families.

In [6]:
run_model("efficientnet_b2")

Running efficientnet_b2 | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_efficientnet_b2.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Projec

100%|██████████| 35.2M/35.2M [00:03<00:00, 11.7MB/s]
                                                                             

efficientnet_b2 epoch=01 loss=2.7469 train_acc=0.2276 val_acc=0.3640 val_macro_f1=0.3357


efficientnet_b2 epoch=02 loss=2.1957 train_acc=0.4344 val_acc=0.4962 val_macro_f1=0.4711


efficientnet_b2 epoch=03 loss=1.8085 train_acc=0.5738 val_acc=0.5805 val_macro_f1=0.5581


efficientnet_b2 epoch=04 loss=1.5400 train_acc=0.6678 val_acc=0.6169 val_macro_f1=0.5940


... (4 outputs left out)

efficientnet_b2 epoch=09 loss=0.8143 train_acc=0.9278 val_acc=0.7011 val_macro_f1=0.6787


efficientnet_b2 epoch=10 loss=0.7551 train_acc=0.9561 val_acc=0.7050 val_macro_f1=0.6825
{
  "model": "efficientnet_b2",
  "val": {
    "accuracy": 0.7049808429118773,
    "macro_f1": 0.6825260940310077,
    "weighted_f1": 0.7003792313579336
  },
  "gap": 0.25113072394620306,
  "overfit_flag": false
}
{
  "selected_model": "efficientnet_b2",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,EfficientNet,efficientnet_b2,complete,"7,729,174",10,0.7050,0.6825,0.7004,0.9561,0.2511,False,518.1,family_walkthrough_efficientnet_b2.pt


{'name': 'efficientnet_b2',
 'settings': {'family': 'efficientnet',
  'backbone': 'efficientnet_b2',
  'lr': 0.001,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 7729174,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.7469123594121996,
    'train_accuracy': 0.2276456111566858,
    'val_accuracy': 0.36398467432950193,
    'val_macro_f1': 0.3357496598641179,
    'val_weighted_f1': 0.34695583679786435},
   {'epoch': 2,
    'train_loss': 2.195691404778973,
    'train_accuracy': 0.43437243642329776,
    'val_accuracy': 0.49616858237547895,
    'val_macro_f1': 0.4710641911815442,
    'val_weighted_f1': 0.48213268322091596},
   {'epoch': 3,
    'train_loss': 1.8084660417893355,
    'train_accuracy': 0.57383100902379,
  

### EfficientNet-V2-S — modern efficient CNN

This small-to-medium V2 model checks whether newer blocks and training design improve transfer on the noisy fixed split.

In [7]:
run_model("efficientnet_v2_s")

Running efficientnet_v2_s | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_efficientnet_v2_s.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Pr

100%|██████████| 82.7M/82.7M [00:09<00:00, 8.98MB/s]
                                                                               

efficientnet_v2_s epoch=01 loss=2.7820 train_acc=0.2133 val_acc=0.3563 val_macro_f1=0.3266


efficientnet_v2_s epoch=02 loss=2.1790 train_acc=0.4368 val_acc=0.5211 val_macro_f1=0.4998


efficientnet_v2_s epoch=03 loss=1.6778 train_acc=0.6120 val_acc=0.6398 val_macro_f1=0.6232


efficientnet_v2_s epoch=04 loss=1.3178 train_acc=0.7473 val_acc=0.6398 val_macro_f1=0.6211


... (4 outputs left out)

efficientnet_v2_s epoch=09 loss=0.6527 train_acc=0.9742 val_acc=0.6935 val_macro_f1=0.6734


efficientnet_v2_s epoch=10 loss=0.6354 train_acc=0.9815 val_acc=0.6897 val_macro_f1=0.6711
{
  "model": "efficientnet_v2_s",
  "val": {
    "accuracy": 0.7183908045977011,
    "macro_f1": 0.6992897123654356,
    "weighted_f1": 0.713191440823756
  },
  "gap": 0.24920558588630215,
  "overfit_flag": false
}
{
  "selected_model": "efficientnet_v2_s",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,EfficientNet,efficientnet_v2_s,complete,"20,203,108",8,0.7184,0.6993,0.7132,0.9676,0.2492,False,570.9,family_walkthrough_efficientnet_v2_s.pt


{'name': 'efficientnet_v2_s',
 'settings': {'family': 'efficientnet_v2',
  'backbone': 'efficientnet_v2_s',
  'lr': 0.0008,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 20203108,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.781954492618835,
    'train_accuracy': 0.2132895816242822,
    'val_accuracy': 0.3563218390804598,
    'val_macro_f1': 0.32663181942480923,
    'val_weighted_f1': 0.3348823701898024},
   {'epoch': 2,
    'train_loss': 2.1789884933459214,
    'train_accuracy': 0.4368334700574241,
    'val_accuracy': 0.5210727969348659,
    'val_macro_f1': 0.4997523274675874,
    'val_weighted_f1': 0.5124647570072841},
   {'epoch': 3,
    'train_loss': 1.6778274423935835,
    'train_accuracy': 0.6119770303527

### EfficientNet-V2-M — larger efficient CNN

This larger V2 candidate tests whether additional efficient capacity recovers difficult classes without crossing the overfit guard.

In [8]:
run_model("efficientnet_v2_m")

Running efficientnet_v2_m | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_efficientnet_v2_m.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Pr

100%|██████████| 208M/208M [00:19<00:00, 10.9MB/s] 
                                                                               

efficientnet_v2_m epoch=01 loss=2.8870 train_acc=0.1620 val_acc=0.3218 val_macro_f1=0.2788


efficientnet_v2_m epoch=02 loss=2.3130 train_acc=0.3893 val_acc=0.5575 val_macro_f1=0.5266


efficientnet_v2_m epoch=03 loss=1.6798 train_acc=0.6038 val_acc=0.6303 val_macro_f1=0.6185


efficientnet_v2_m epoch=04 loss=1.2796 train_acc=0.7518 val_acc=0.6628 val_macro_f1=0.6425


... (4 outputs left out)

efficientnet_v2_m epoch=09 loss=0.6540 train_acc=0.9779 val_acc=0.7031 val_macro_f1=0.6847


efficientnet_v2_m epoch=10 loss=0.6293 train_acc=0.9865 val_acc=0.6992 val_macro_f1=0.6764
{
  "model": "efficientnet_v2_m",
  "val": {
    "accuracy": 0.7222222222222222,
    "macro_f1": 0.7019878746097092,
    "weighted_f1": 0.7204168276970518
  },
  "gap": 0.22855710509525118,
  "overfit_flag": false
}
{
  "selected_model": "efficientnet_v2_m",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,EfficientNet,efficientnet_v2_m,complete,"52,883,976",7,0.7222,0.7020,0.7204,0.9508,0.2286,False,739.0,family_walkthrough_efficientnet_v2_m.pt


{'name': 'efficientnet_v2_m',
 'settings': {'family': 'efficientnet_v2',
  'backbone': 'efficientnet_v2_m',
  'lr': 0.0006,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 52883976,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.8870287814171487,
    'train_accuracy': 0.1620180475799836,
    'val_accuracy': 0.3218390804597701,
    'val_macro_f1': 0.2787541268803629,
    'val_weighted_f1': 0.29819390242866206},
   {'epoch': 2,
    'train_loss': 2.3129565474254634,
    'train_accuracy': 0.389253486464315,
    'val_accuracy': 0.5574712643678161,
    'val_macro_f1': 0.5266127759044652,
    'val_weighted_f1': 0.543403378931059},
   {'epoch': 3,
    'train_loss': 1.67978727350048,
    'train_accuracy': 0.6037735849056604

## MobileNet family

MobileNet uses lightweight depthwise-separable convolutions and mobile-oriented blocks to reduce inference cost.

**Pros:** fast, compact, and practical for low-resource deployment.

**Cons:** lower capacity can hurt fine-grained or noisy classes and leaves less room for label ambiguity.

### MobileNet-V3-Large — lightweight CNN

This mobile-oriented model measures the accuracy cost of deployment-friendly design rather than maximum capacity.

In [9]:
run_model("mobilenet_v3_large")

Running mobilenet_v3_large | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_mobilenet_v3_large.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-

100%|██████████| 21.1M/21.1M [00:02<00:00, 10.6MB/s]
                                                                                

mobilenet_v3_large epoch=01 loss=2.4827 train_acc=0.3154 val_acc=0.4559 val_macro_f1=0.4444


mobilenet_v3_large epoch=02 loss=1.8246 train_acc=0.5537 val_acc=0.5441 val_macro_f1=0.5242


mobilenet_v3_large epoch=03 loss=1.5351 train_acc=0.6575 val_acc=0.5843 val_macro_f1=0.5698


mobilenet_v3_large epoch=04 loss=1.3263 train_acc=0.7330 val_acc=0.5996 val_macro_f1=0.5853


... (3 outputs left out)

mobilenet_v3_large epoch=08 loss=0.8354 train_acc=0.9299 val_acc=0.6513 val_macro_f1=0.6231


mobilenet_v3_large epoch=09 loss=0.7864 train_acc=0.9495 val_acc=0.6686 val_macro_f1=0.6467
{
  "model": "mobilenet_v3_large",
  "val": {
    "accuracy": 0.6704980842911877,
    "macro_f1": 0.6476144512847939,
    "weighted_f1": 0.6679807606215475
  },
  "gap": 0.18553144811242173,
  "overfit_flag": false
}
{
  "selected_model": "mobilenet_v3_large",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,MobileNet,mobilenet_v3_large,complete,"4,227,652",6,0.6705,0.6476,0.6680,0.8560,0.1855,False,385.2,family_walkthrough_mobilenet_v3_large.pt


{'name': 'mobilenet_v3_large',
 'settings': {'family': 'mobilenet',
  'backbone': 'mobilenet_v3_large',
  'lr': 0.001,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 4227652,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.48270331177057,
    'train_accuracy': 0.31542247744052504,
    'val_accuracy': 0.4559386973180077,
    'val_macro_f1': 0.44444480226099764,
    'val_weighted_f1': 0.45389668071917133},
   {'epoch': 2,
    'train_loss': 1.8246322776757034,
    'train_accuracy': 0.5537325676784249,
    'val_accuracy': 0.5440613026819924,
    'val_macro_f1': 0.5241620642701542,
    'val_weighted_f1': 0.5411710147027451},
   {'epoch': 3,
    'train_loss': 1.5350727622025933,
    'train_accuracy': 0.6575061525840853,


## ConvNeXt family

ConvNeXt is a modernized pure-convolutional family that adopts several design and training ideas associated with vision transformers.

**Pros:** strong pretrained transfer, competitive accuracy, simple convolutional inference, and a clean size ladder.

**Cons:** larger variants are memory-heavy and can amplify dataset noise without regularisation.

### ConvNeXt-Tiny — modern CNN baseline

This smallest ConvNeXt candidate tests whether modernised convolutional blocks transfer better than older CNN families.

In [10]:
run_model("convnext_tiny")

Running convnext_tiny | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_convnext_tiny.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/no

100%|██████████| 109M/109M [00:11<00:00, 9.58MB/s] 
                                                                           

convnext_tiny epoch=01 loss=2.6256 train_acc=0.2900 val_acc=0.4291 val_macro_f1=0.4045


convnext_tiny epoch=02 loss=1.9553 train_acc=0.5427 val_acc=0.5824 val_macro_f1=0.5643


convnext_tiny epoch=03 loss=1.4670 train_acc=0.7190 val_acc=0.6705 val_macro_f1=0.6514


convnext_tiny epoch=04 loss=1.1645 train_acc=0.8285 val_acc=0.7126 val_macro_f1=0.6966


... (4 outputs left out)

convnext_tiny epoch=09 loss=0.6141 train_acc=0.9910 val_acc=0.7471 val_macro_f1=0.7290


convnext_tiny epoch=10 loss=0.5959 train_acc=0.9881 val_acc=0.7529 val_macro_f1=0.7354
{
  "model": "convnext_tiny",
  "val": {
    "accuracy": 0.7528735632183908,
    "macro_f1": 0.7353660387768514,
    "weighted_f1": 0.7521765655297369
  },
  "gap": 0.23523144088333192,
  "overfit_flag": false
}
{
  "selected_model": "convnext_tiny",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ConvNeXt,convnext_tiny,complete,"27,835,508",10,0.7529,0.7354,0.7522,0.9881,0.2352,False,651.4,family_walkthrough_convnext_tiny.pt


{'name': 'convnext_tiny',
 'settings': {'family': 'convnext',
  'backbone': 'convnext_tiny',
  'lr': 0.0005,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 27835508,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.6256273805705552,
    'train_accuracy': 0.2899917965545529,
    'val_accuracy': 0.42911877394636017,
    'val_macro_f1': 0.4045025317156264,
    'val_weighted_f1': 0.41481548366850746},
   {'epoch': 2,
    'train_loss': 1.955321286238876,
    'train_accuracy': 0.5426579163248565,
    'val_accuracy': 0.5823754789272031,
    'val_macro_f1': 0.5642538546940251,
    'val_weighted_f1': 0.576948447830177},
   {'epoch': 3,
    'train_loss': 1.466980976996079,
    'train_accuracy': 0.7190319934372437,
    'val_ac

### ConvNeXt-Small — medium modern CNN

This middle ConvNeXt size checks whether extra modern-CNN capacity improves macro-F1 without fitting dataset noise.

In [11]:
run_model("convnext_small")

Running convnext_small | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_convnext_small.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/

100%|██████████| 192M/192M [00:17<00:00, 11.3MB/s] 
                                                                            

convnext_small epoch=01 loss=2.7097 train_acc=0.2551 val_acc=0.4234 val_macro_f1=0.3870


convnext_small epoch=02 loss=2.0630 train_acc=0.5008 val_acc=0.5900 val_macro_f1=0.5595


convnext_small epoch=03 loss=1.6024 train_acc=0.6571 val_acc=0.6379 val_macro_f1=0.6169


convnext_small epoch=04 loss=1.3153 train_acc=0.7695 val_acc=0.6743 val_macro_f1=0.6527


... (4 outputs left out)

convnext_small epoch=09 loss=0.6514 train_acc=0.9861 val_acc=0.7356 val_macro_f1=0.7187


convnext_small epoch=10 loss=0.6310 train_acc=0.9840 val_acc=0.7395 val_macro_f1=0.7234
{
  "model": "convnext_small",
  "val": {
    "accuracy": 0.7394636015325671,
    "macro_f1": 0.7234035373395961,
    "weighted_f1": 0.735666919931011
  },
  "gap": 0.24453967984561176,
  "overfit_flag": false
}
{
  "selected_model": "convnext_small",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ConvNeXt,convnext_small,complete,"49,470,068",10,0.7395,0.7234,0.7357,0.9840,0.2445,False,868.1,family_walkthrough_convnext_small.pt


{'name': 'convnext_small',
 'settings': {'family': 'convnext',
  'backbone': 'convnext_small',
  'lr': 0.0004,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 49470068,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.7096533136430128,
    'train_accuracy': 0.2551271534044299,
    'val_accuracy': 0.42337164750957856,
    'val_macro_f1': 0.38699461794356876,
    'val_weighted_f1': 0.40017435207903224},
   {'epoch': 2,
    'train_loss': 2.062986089513193,
    'train_accuracy': 0.5008203445447088,
    'val_accuracy': 0.5900383141762452,
    'val_macro_f1': 0.5595312835947749,
    'val_weighted_f1': 0.5758532110371616},
   {'epoch': 3,
    'train_loss': 1.602396252887701,
    'train_accuracy': 0.6570959803117309,
    'va

### ConvNeXt-Base — large modern CNN

This largest ConvNeXt tests whether more pretrained capacity improves difficult classes under regularised training.

In [12]:
run_model("convnext_base")

Running convnext_base | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_convnext_base.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/no

100%|██████████| 338M/338M [00:31<00:00, 11.3MB/s] 
                                                                           

convnext_base epoch=01 loss=2.7513 train_acc=0.2502 val_acc=0.4080 val_macro_f1=0.3731


convnext_base epoch=02 loss=2.2381 train_acc=0.4733 val_acc=0.5211 val_macro_f1=0.4861


convnext_base epoch=03 loss=1.8540 train_acc=0.6042 val_acc=0.5881 val_macro_f1=0.5556


convnext_base epoch=04 loss=1.5762 train_acc=0.6969 val_acc=0.6475 val_macro_f1=0.6267


... (4 outputs left out)

convnext_base epoch=09 loss=0.7857 train_acc=0.9537 val_acc=0.7318 val_macro_f1=0.7161


convnext_base epoch=10 loss=0.7093 train_acc=0.9705 val_acc=0.7395 val_macro_f1=0.7214
{
  "model": "convnext_base",
  "val": {
    "accuracy": 0.7394636015325671,
    "macro_f1": 0.7213900576379447,
    "weighted_f1": 0.7362853553060077
  },
  "gap": 0.2310039948579169,
  "overfit_flag": false
}
{
  "selected_model": "convnext_base",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ConvNeXt,convnext_base,complete,"87,586,964",10,0.7395,0.7214,0.7363,0.9705,0.2310,False,1114.4,family_walkthrough_convnext_base.pt


{'name': 'convnext_base',
 'settings': {'family': 'convnext',
  'backbone': 'convnext_base',
  'lr': 0.0003,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 87586964,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.7512751856660533,
    'train_accuracy': 0.2502050861361772,
    'val_accuracy': 0.40804597701149425,
    'val_macro_f1': 0.3730522694791247,
    'val_weighted_f1': 0.3831666058832212},
   {'epoch': 2,
    'train_loss': 2.238081727152556,
    'train_accuracy': 0.47333880229696473,
    'val_accuracy': 0.5210727969348659,
    'val_macro_f1': 0.4861262607929529,
    'val_weighted_f1': 0.5007663332857952},
   {'epoch': 3,
    'train_loss': 1.8540110182917975,
    'train_accuracy': 0.6041837571780148,
    'val_

## Swin family

Swin is a hierarchical vision transformer that uses local attention windows and shifts those windows between blocks.

**Pros:** multi-scale representations and broader spatial context than a conventional local convolution.

**Cons:** higher compute and memory cost, more training sensitivity, and no guarantee that attention helps a small noisy dataset.

### Swin-T — compact windowed transformer

This compact transformer baseline tests whether local-window attention and hierarchy beat convolutional inductive bias.

In [13]:
run_model("swin_t")

Running swin_t | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_swin_t.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/output

100%|██████████| 108M/108M [00:10<00:00, 11.3MB/s] 
                                                                    

swin_t epoch=01 loss=2.6891 train_acc=0.2625 val_acc=0.4406 val_macro_f1=0.4117


swin_t epoch=02 loss=2.0447 train_acc=0.4815 val_acc=0.5594 val_macro_f1=0.5430


swin_t epoch=03 loss=1.5897 train_acc=0.6317 val_acc=0.6456 val_macro_f1=0.6303


swin_t epoch=04 loss=1.3081 train_acc=0.7395 val_acc=0.6533 val_macro_f1=0.6375


... (4 outputs left out)

swin_t epoch=09 loss=0.7548 train_acc=0.9512 val_acc=0.7184 val_macro_f1=0.6960


swin_t epoch=10 loss=0.7122 train_acc=0.9639 val_acc=0.7203 val_macro_f1=0.6950
{
  "model": "swin_t",
  "val": {
    "accuracy": 0.7183908045977011,
    "macro_f1": 0.6989850835868743,
    "weighted_f1": 0.718358344742817
  },
  "gap": 0.190961123211979,
  "overfit_flag": false
}
{
  "selected_model": "swin_t",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,Swin,swin_t,complete,"27,534,734",7,0.7184,0.6990,0.7184,0.9094,0.1910,False,648.3,family_walkthrough_swin_t.pt


{'name': 'swin_t',
 'settings': {'family': 'swin',
  'backbone': 'swin_t',
  'lr': 0.0005,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 27534734,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.6890705301870708,
    'train_accuracy': 0.26251025430680885,
    'val_accuracy': 0.44061302681992337,
    'val_macro_f1': 0.4116809628162141,
    'val_weighted_f1': 0.42494503805844336},
   {'epoch': 2,
    'train_loss': 2.0446540078306508,
    'train_accuracy': 0.4815422477440525,
    'val_accuracy': 0.5593869731800766,
    'val_macro_f1': 0.5429817856830805,
    'val_weighted_f1': 0.5545498478598614},
   {'epoch': 3,
    'train_loss': 1.589693942101173,
    'train_accuracy': 0.6316652994257588,
    'val_accuracy': 0.6455

### Swin-V2-T — updated compact transformer

This similarly compact V2 model separates the updated transformer design from simply buying a larger model.

In [14]:
run_model("swin_v2_t")

Running swin_v2_t | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_swin_v2_t.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/

  0%|          | 128k/109M [00:00<02:18, 822kB/s]

Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /home/vaper/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth


100%|██████████| 109M/109M [00:11<00:00, 9.69MB/s] 
                                                                       

swin_v2_t epoch=01 loss=2.6740 train_acc=0.2748 val_acc=0.4004 val_macro_f1=0.3639


swin_v2_t epoch=02 loss=2.0559 train_acc=0.4787 val_acc=0.5268 val_macro_f1=0.5171


... (6 outputs left out)

swin_v2_t epoch=09 loss=0.7443 train_acc=0.9516 val_acc=0.7031 val_macro_f1=0.6874


swin_v2_t epoch=10 loss=0.7086 train_acc=0.9643 val_acc=0.7222 val_macro_f1=0.7025
{
  "model": "swin_v2_t",
  "val": {
    "accuracy": 0.7222222222222222,
    "macro_f1": 0.7024885723152072,
    "weighted_f1": 0.7184808998766309
  },
  "gap": 0.24209279008294593,
  "overfit_flag": false
}
{
  "selected_model": "swin_v2_t",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,Swin,swin_v2_t,complete,"27,597,950",10,0.7222,0.7025,0.7185,0.9643,0.2421,False,710.6,family_walkthrough_swin_v2_t.pt


{'name': 'swin_v2_t',
 'settings': {'family': 'swin_v2',
  'backbone': 'swin_v2_t',
  'lr': 0.0005,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 27597950,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.6740272512622907,
    'train_accuracy': 0.2748154224774405,
    'val_accuracy': 0.4003831417624521,
    'val_macro_f1': 0.3639265474052189,
    'val_weighted_f1': 0.37555178909546016},
   {'epoch': 2,
    'train_loss': 2.055879223190881,
    'train_accuracy': 0.47867104183757175,
    'val_accuracy': 0.5268199233716475,
    'val_macro_f1': 0.5170536524206919,
    'val_weighted_f1': 0.5254256645187355},
   {'epoch': 3,
    'train_loss': 1.581867484485402,
    'train_accuracy': 0.6423297785069729,
    'val_accuracy':

### Swin-B — large windowed transformer

This largest transformer candidate tests the upper end of attention-based capacity with validation-only selection.

In [15]:
run_model("swin_b")

Running swin_b | recipe=robust-v2-tta | profile=standard | report=model_walkthrough_robust_v2_tta_swin_b.json
cwd=/home/vaper/Work/Prototypes/626-Project/notebooks
device=cuda  batch_size=16  num_workers=0
data/raw → /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw
╭──────────────────── Component classification ─────────────────────╮
│ profile=standard · device=cuda · image=224px · batch=16           │
│ work=/home/vaper/Work/Prototypes/626-Project/notebooks            │
│ data=/home/vaper/Work/Prototypes/626-Project/notebooks/data       │
│ outputs=/home/vaper/Work/Prototypes/626-Project/notebooks/outputs │
╰───────────────────────────────────────────────────────────────────╯
Using existing dataset at /home/vaper/Work/Prototypes/626-Project/notebooks/data/raw/electronic-component-recognizer
Loaded frozen split from 
/home/vaper/Work/Prototypes/626-Project/notebooks/data/splits/split_manifest_raw
.json
session 
manifest=/home/vaper/Work/Prototypes/626-Project/notebooks/output

100%|██████████| 335M/335M [00:33<00:00, 10.4MB/s] 
                                                                    

swin_b epoch=01 loss=2.7021 train_acc=0.2777 val_acc=0.4387 val_macro_f1=0.4098


swin_b epoch=02 loss=2.0636 train_acc=0.4902 val_acc=0.5421 val_macro_f1=0.5263


swin_b epoch=03 loss=1.6401 train_acc=0.6272 val_acc=0.6015 val_macro_f1=0.5830


swin_b epoch=04 loss=1.3422 train_acc=0.7342 val_acc=0.6705 val_macro_f1=0.6555


... (4 outputs left out)

swin_b epoch=09 loss=0.7446 train_acc=0.9549 val_acc=0.7146 val_macro_f1=0.6994


swin_b epoch=10 loss=0.7022 train_acc=0.9672 val_acc=0.7356 val_macro_f1=0.7205
{
  "model": "swin_b",
  "val": {
    "accuracy": 0.735632183908046,
    "macro_f1": 0.720462554956877,
    "weighted_f1": 0.7347706474714139
  },
  "gap": 0.23155403430360288,
  "overfit_flag": false
}
{
  "selected_model": "swin_b",
  "selected_test_after_validation_selection": null,
  "status": "validation_complete_test_deferred"
}


,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,Swin,swin_b,complete,"86,763,724",10,0.7356,0.7205,0.7348,0.9672,0.2316,False,1090.3,family_walkthrough_swin_b.pt


{'name': 'swin_b',
 'settings': {'family': 'swin',
  'backbone': 'swin_b',
  'lr': 0.0004,
  'weight_decay': 0.0001,
  'freeze_backbone_epochs': 1,
  'epochs': 10,
  'patience': 3,
  'selection': 'raw_val_macro_f1',
  'training_variant': 'robust-v2-tta',
  'label_smoothing': 0.08,
  'class_weight_power': 0.5,
  'training_exclusions_sha256': None,
  'training_exclusions_applied_count': 0},
 'params': 86763724,
 'training': {'history': [{'epoch': 1,
    'train_loss': 2.7021207357543746,
    'train_accuracy': 0.2776866283839213,
    'val_accuracy': 0.4386973180076628,
    'val_macro_f1': 0.4097978843213353,
    'val_weighted_f1': 0.4241745418487775},
   {'epoch': 2,
    'train_loss': 2.0636423647014146,
    'train_accuracy': 0.49015586546349466,
    'val_accuracy': 0.5421455938697318,
    'val_macro_f1': 0.526333654982054,
    'val_weighted_f1': 0.5395461242717592},
   {'epoch': 3,
    'train_loss': 1.6400899466346293,
    'train_accuracy': 0.6271534044298606,
    'val_accuracy': 0.601532

In [16]:
# Compile all per-model results into one ordered table and save it for later review.
rows = []
for model in MODEL_ORDER:
    if model in MODEL_RECORDS:
        rows.append(_row(MODEL_RECORDS[model]))
    else:
        rows.append({
            "family": MODEL_FAMILY[model],
            "model": model,
            "status": "not run",
            "parameters": float("nan"),
            "best_epoch": float("nan"),
            "val_accuracy": float("nan"),
            "val_macro_f1": float("nan"),
            "val_weighted_f1": float("nan"),
            "train_accuracy": float("nan"),
            "train_val_gap": float("nan"),
            "overfit_flag": None,
            "wall_time_seconds": float("nan"),
            "checkpoint": "",
        })

summary = pd.DataFrame(rows, columns=[
    "family", "model", "status", "parameters", "best_epoch",
    "val_accuracy", "val_macro_f1", "val_weighted_f1",
    "train_accuracy", "train_val_gap", "overfit_flag",
    "wall_time_seconds", "checkpoint",
])
display(summary.style.format({
    "parameters": lambda value: "—" if pd.isna(value) else f"{int(value):,}",
    "best_epoch": lambda value: "—" if pd.isna(value) else f"{int(value)}",
    "val_accuracy": _format_number,
    "val_macro_f1": _format_number,
    "val_weighted_f1": _format_number,
    "train_accuracy": _format_number,
    "train_val_gap": _format_number,
    "wall_time_seconds": lambda value: "—" if pd.isna(value) else f"{value:.1f}",
}).background_gradient(subset=["val_accuracy", "val_macro_f1"], cmap="YlGn"))

summary_dir = NOTEBOOK_DIR / "outputs" / "metrics"
summary_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_dir / f"{REPORT_PREFIX}_summary.csv", index=False)
summary.to_json(summary_dir / f"{REPORT_PREFIX}_summary.json", orient="records", indent=2)

complete = summary[summary["status"] == "complete"]
missing = summary.loc[summary["status"] != "complete", "model"].tolist()
if missing:
    print(f"Run the remaining model cells before treating the summary as complete: {missing}")
if not complete.empty:
    family_summary = (
        complete.groupby("family", sort=False)
        .agg(
            models=("model", "count"),
            best_val_accuracy=("val_accuracy", "max"),
            best_val_macro_f1=("val_macro_f1", "max"),
            mean_train_val_gap=("train_val_gap", "mean"),
        )
        .reset_index()
    )
    display(family_summary.style.format({
        "best_val_accuracy": _format_number,
        "best_val_macro_f1": _format_number,
        "mean_train_val_gap": _format_number,
    }).background_gradient(subset=["best_val_accuracy", "best_val_macro_f1"], cmap="YlGn"))
print(f"Saved {len(complete)}/{len(summary)} model rows to {summary_dir / (REPORT_PREFIX + '_summary.csv')}")

,family,model,status,parameters,best_epoch,val_accuracy,val_macro_f1,val_weighted_f1,train_accuracy,train_val_gap,overfit_flag,wall_time_seconds,checkpoint
0,ResNet,resnet34,complete,"21,294,932",5,0.6590,0.6317,0.6547,0.8843,0.2253,False,381.2,family_walkthrough_resnet34.pt
1,ResNet,resnet101,complete,"42,541,140",8,0.7299,0.7100,0.7271,0.9861,0.2562,False,652.3,family_walkthrough_resnet101.pt
2,ResNet,resnet152,complete,"58,184,788",9,0.7241,0.7040,0.7166,0.9906,0.2664,False,775.9,family_walkthrough_resnet152.pt
3,DenseNet,densenet121,complete,"6,974,356",8,0.6877,0.6665,0.6875,0.9639,0.2762,False,560.2,family_walkthrough_densenet121.pt
4,EfficientNet,efficientnet_b2,complete,"7,729,174",10,0.7050,0.6825,0.7004,0.9561,0.2511,False,518.1,family_walkthrough_efficientnet_b2.pt
5,EfficientNet,efficientnet_v2_s,complete,"20,203,108",8,0.7184,0.6993,0.7132,0.9676,0.2492,False,570.9,family_walkthrough_efficientnet_v2_s.pt
6,EfficientNet,efficientnet_v2_m,complete,"52,883,976",7,0.7222,0.7020,0.7204,0.9508,0.2286,False,739.0,family_walkthrough_efficientnet_v2_m.pt
7,MobileNet,mobilenet_v3_large,complete,"4,227,652",6,0.6705,0.6476,0.6680,0.8560,0.1855,False,385.2,family_walkthrough_mobilenet_v3_large.pt
8,ConvNeXt,convnext_tiny,complete,"27,835,508",10,0.7529,0.7354,0.7522,0.9881,0.2352,False,651.4,family_walkthrough_convnext_tiny.pt
9,ConvNeXt,convnext_small,complete,"49,470,068",10,0.7395,0.7234,0.7357,0.9840,0.2445,False,868.1,family_walkthrough_convnext_small.pt


,family,models,best_val_accuracy,best_val_macro_f1,mean_train_val_gap
0,ResNet,3,0.7299,0.7100,0.2493
1,DenseNet,1,0.6877,0.6665,0.2762
2,EfficientNet,3,0.7222,0.7020,0.2430
3,MobileNet,1,0.6705,0.6476,0.1855
4,ConvNeXt,3,0.7529,0.7354,0.2369
5,Swin,3,0.7356,0.7205,0.2215


Saved 14/14 model rows to /home/vaper/Work/Prototypes/626-Project/notebooks/outputs/metrics/model_walkthrough_robust_v2_tta_summary.csv
